# 007 - JupyterLab Sidebar Chatbot 데모

JupyterLab **우측 사이드바**에 뜨는 챗봇입니다. **jupyter 서버를 재시작할 수 없는 환경**(회사 폐쇄망)을 위해 이렇게 동작합니다.

1. 프론트엔드 labextension(우측 💬 탭) → 설치 후 **브라우저 새로고침**만으로 뜸 (서버 재시작 X)
2. 챗봇 두뇌 → **이 노트북 셀에서 `start_brain_server()` 한 줄**로 localhost 에 띄움 (커널은 내가 실행 가능)

> ⚠️ 전제: 브라우저의 `127.0.0.1` 과 이 커널이 **같은 기계**여야 합니다(로컬/컨테이너/SSH 포트포워딩). single-file `.py` 가 아니라 빌드가 필요한 익스텐션입니다 — 자세한 건 `README.md` 참고.

## 1) 두뇌만 빠르게 체험 (서버 없이)

사이드바가 호출하는 두뇌(`jlab_sidebar_chatbot/llm.py`)는 표준 라이브러리만 써서 바로 import 됩니다.

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))  # 설치 없이 import 하려고 경로 추가

from jlab_sidebar_chatbot import ChatBrain

brain = ChatBrain(system_prompt='너는 폐쇄망 데모 도우미야.')
sid = 'notebook-demo'
for line in ['안녕하세요', '이거 어떻게 쓰나요?', '고맙습니다']:
    ans = brain.send(sid, line)
    print(f'🙂 user: {line}')
    print(f'🤖 bot : {ans["content"]}\n')

🙂 user: 안녕하세요
🤖 bot : 안녕하세요! 무엇을 도와드릴까요? (저는 외부 모델 없이 동작하는 데모 챗봇입니다)

🙂 user: 이거 어떻게 쓰나요?
🤖 bot : "이거 어떻게 쓰나요?" 라고 물어보셨네요. 지금은 Mock 응답이라 실제 답을 드리진 못하지만, 사내 LLM 어댑터를 연결하면 여기로 진짜 답변이 들어옵니다.

🙂 user: 고맙습니다
🤖 bot : [3번째 턴] 방금 "고맙습니다" 라고 하셨습니다. 계속 말씀해 주세요.



## 2) 사이드바 탭에 두뇌 붙이기 — 이 셀 한 줄

아래 셀을 실행하면 커널 안에서 localhost HTTP 서버(기본 포트 8765)가 백그라운드로 뜹니다. 그러면 우측 💬 탭에서 보낸 메시지가 이 서버로 전달됩니다. **jupyter 서버 재시작은 전혀 필요 없습니다.**

In [4]:
from jlab_sidebar_chatbot import start_brain_server

start_brain_server()   # → http://127.0.0.1:8765 (백그라운드 스레드)
# 이제 오른쪽 사이드바의 💬 탭을 열고 대화해 보세요.
# 중지하려면:  from jlab_sidebar_chatbot import stop_brain_server; stop_brain_server()

✅ 챗봇 서버 시작: http://127.0.0.1:8765  — 우측 💬 탭에서 대화하세요.


## 3) 설치 (폐쇄망, 서버 재시작 없이)

dev 환경에서 wheel 을 만들어 반입한 뒤:

```bash
pip install jlab_sidebar_chatbot-*.whl   # jupyter 가 쓰는 env 에
```

그다음 **브라우저에서 JupyterLab 페이지를 새로고침**하면 우측에 💬 탭이 생깁니다(서버 재시작 X). 마지막으로 위 2) 셀을 실행하면 대화가 동작합니다.

> 왜 새로고침만으로 되나: jupyter 서버는 페이지를 그릴 때마다 labextensions 폴더를 다시 스캔합니다. 반면 '서버 익스텐션'으로 만든 백엔드는 서버 재시작 때만 등록되므로, 두뇌를 셀 서버로 분리한 것입니다.

## 4) 사내 LLM 으로 교체

`LLMAdapter` 를 구현해 `start_brain_server(brain=...)` 로 주입합니다.

```python
from jlab_sidebar_chatbot import start_brain_server, ChatBrain
from jlab_sidebar_chatbot.llm import LLMAdapter

class MyLocalLLM(LLMAdapter):
    def generate(self, messages):
        # messages: [{'role': 'user'/'assistant'/'system', 'content': ...}, ...]
        # → 사내 모델 호출 후 답변 문자열만 반환
        ...

start_brain_server(brain=ChatBrain(adapter=MyLocalLLM()))
```

> HTTP 로 사내 추론 서버를 호출한다면 `requests`/`httpx` 대신 표준 `urllib.request` 를 쓰세요.